# Programming the Robot
## Sequence, Iteration, and Selection

This lesson runs alongside the **Robot Logic Game**. Each section introduces one programming concept, shows what it looks like in Python, and points you to a level to practice it in the game.

**Launch the game** in a separate terminal:

```bash
make -C _projects/robot-logic-game serve
```

Then open <http://localhost:8765/test.html> and keep it side-by-side with this notebook.

---

### What you'll learn

Every program — from a calculator to a search engine — is built from three patterns:

1. **Sequence** — do these instructions in order.
2. **Iteration (loops)** — do something N times, or until a condition changes.
3. **Selection (conditionals)** — do something *only if* a condition is true.

The game gives each pattern a draggable block. Python gives each pattern a keyword. Same idea, different costume.

## 1. What is a program?

A **program** is an ordered list of instructions a computer follows from top to bottom.

In the game, a program is the stack of blocks in the **Program** panel. The robot reads from top to bottom and executes each block in order.

The robot's instruction set:

| Block | What it does |
| --- | --- |
| `Move Forward` | Move one tile in the direction the robot is facing. |
| `Move Backward` | Move one tile *behind* the robot, without turning. |
| `Turn Left` | Rotate 90° counter-clockwise. |
| `Turn Right` | Rotate 90° clockwise. |
| `Repeat N times` | Loop — run the body N times. |
| `If Can Move Forward` | Conditional — run the body only if the path ahead is clear. |

**Try this now:** open **Level 1: First Steps** in the game. Drag a single `Move Forward` block into the Program panel and press **Run**. The robot moves one tile. That's the smallest possible program.

## 2. Sequence — one step at a time

When you stack blocks, the robot does them top-to-bottom. Nothing fancy — just one after another.

Let's build a tiny Python simulation of the robot so we can experiment without the game. The class below mirrors the game's engine: it tracks position and facing direction and prints what it does after each move.

In [ ]:
class Robot:
    DIRS = [(1, 0), (0, 1), (-1, 0), (0, -1)]   # East, South, West, North
    NAMES = ['East', 'South', 'West', 'North']

    def __init__(self, col=0, row=3, facing=0, walls=None, grid_size=7):
        self.col = col
        self.row = row
        self.facing = facing
        self.walls = set(walls or [])
        self.grid_size = grid_size
        self.steps = 0

    def can_move_forward(self):
        dx, dy = self.DIRS[self.facing]
        nc, nr = self.col + dx, self.row + dy
        if not (0 <= nc < self.grid_size and 0 <= nr < self.grid_size):
            return False
        return (nc, nr) not in self.walls

    def forward(self):
        dx, dy = self.DIRS[self.facing]
        nc, nr = self.col + dx, self.row + dy
        if not self.can_move_forward():
            print(f'  forward  → BLOCKED at ({self.col}, {self.row})')
            return
        self.col, self.row = nc, nr
        self.steps += 1
        print(f'  forward  → ({self.col}, {self.row}) facing {self.NAMES[self.facing]}')

    def backward(self):
        dx, dy = self.DIRS[self.facing]
        self.col -= dx
        self.row -= dy
        self.steps += 1
        print(f'  backward → ({self.col}, {self.row}) facing {self.NAMES[self.facing]}')

    def turn_left(self):
        self.facing = (self.facing + 3) % 4
        print(f'  turn L   → facing {self.NAMES[self.facing]}')

    def turn_right(self):
        self.facing = (self.facing + 1) % 4
        print(f'  turn R   → facing {self.NAMES[self.facing]}')


print('Robot class ready.')

Here's a sequential program — Level 1's solution with no loops. Notice how repetitive it is.

In [ ]:
# Level 1: start at (0, 3) facing East. Goal is (6, 0).
robot = Robot(col=0, row=3, facing=0)

robot.forward()
robot.forward()
robot.forward()
robot.forward()
robot.forward()
robot.forward()
robot.turn_left()
robot.forward()
robot.forward()
robot.forward()

print()
print(f'Final position: ({robot.col}, {robot.row})')
print(f'Goal reached?   {(robot.col, robot.row) == (6, 0)}')

**Game challenge:** open **Level 1** and replicate this program by dragging 9 forward blocks and 1 turn-left block into the Program panel.

Press Run. You'll get **1 star** — the robot reached the goal but used a lot of blocks. The game will say *"Try condensing repeated blocks with Repeat."*

That's the next pattern.

## 3. Iteration — loops

If you keep writing the same line over and over, there's a better way: a **loop**.

The game's `Repeat N times` block wraps other blocks and runs them N times. In Python, that's a `for` loop. Here's the same program shrunk:

In [ ]:
# Level 1, this time with loops
robot = Robot(col=0, row=3, facing=0)

for _ in range(6):       # game equivalent: Repeat 6 [Forward]
    robot.forward()
robot.turn_left()
for _ in range(3):       # game equivalent: Repeat 3 [Forward]
    robot.forward()

print()
print(f'Final position: ({robot.col}, {robot.row})')

Same path. Half the lines.

| Approach | Blocks (game) | Lines (Python) |
| --- | --- | --- |
| Pure sequence | 10 | 10 |
| With Repeat / `for` | 5 | 5 |

The game scores you on this. **5 blocks earns 3 stars on Level 1.** A program isn't just "does it work" — concise, reusable code is part of the craft.

**Game challenge:** earn 3 stars on Level 1 by using two `Repeat N times` blocks instead of stacking 10 forwards.

**Bigger challenge — Level 3 ("The Long Way"):** the path snakes around a long wall and the optimal program is 11 blocks. Without loops you'd need 19. Try Level 3, count your blocks, then try to shrink it.

### Loops inside loops

Loops can be *nested* — a Repeat inside a Repeat. Useful for repeating a small pattern many times. We'll need this for the final level.

In [ ]:
# Trace a 3x3 square — 4 sides, each side is 3 forwards + 1 turn
robot = Robot(col=0, row=0, facing=0)

for side in range(4):
    for _ in range(3):
        robot.forward()
    robot.turn_right()

print()
print(f'Returned to ({robot.col}, {robot.row}) facing {robot.NAMES[robot.facing]}')

## 4. Selection — conditionals

Real programs make decisions. The game's `If Can Move Forward` block runs its body only when the path ahead is clear.

The Python equivalent is `if`:

```python
if robot.can_move_forward():
    robot.forward()
```

On its own, "if can move, move" only walks one tile — useful but limited. The pattern gets powerful when you put it **inside a loop**.

In [ ]:
# Walk forward until blocked, without knowing in advance how far that is.
# The robot is in a corridor with a wall at column 4. We don't 'count' — we feel our way.
robot = Robot(col=0, row=3, facing=0, walls=[(4, 3)])

for _ in range(20):                  # generous upper bound
    if robot.can_move_forward():
        robot.forward()

print()
print(f'Stopped at ({robot.col}, {robot.row}) — wall blocked further travel.')

Notice what just happened: the robot doesn't know how long the corridor is. It just **tries** to move 20 times, and the `if` quietly skips the move when it would crash.

That's defensive programming — your code adapts to the world instead of assuming.

**Game challenge — Level 6 ("Right-Hand Spiral"):** the optimal solution uses one outer loop, one inner loop, and one `If Can Move Forward` — 5 blocks total. Without conditionals you'd need to count each corridor by hand.

Try this exact pattern in the game:

```
Repeat 3 times:
    Repeat 10 times:
        If Can Move Forward:
            Forward
    Turn Right
```

Read it aloud: *"Three times: walk until blocked, then turn right."* That's the **right-hand rule** — a classic maze-solving algorithm in 5 blocks.

## 5. Putting it together

The three patterns — **sequence, iteration, selection** — combine to express any algorithm. Computer scientists call this the *structured programming theorem*.

Below is a full Python simulation of the Level 6 spiral. Compare the **3 control structures** in code with the **5 blocks** in the game. They're the same algorithm, rendered differently.

In [ ]:
# Level 6: walls at (3, 0) and (2, 3). Start (0, 0) facing East. Goal (0, 2).
robot = Robot(col=0, row=0, facing=0, walls=[(3, 0), (2, 3)])

for _ in range(3):                       # outer: walk-and-turn 3 times
    for _ in range(10):                  # inner: walk until blocked
        if robot.can_move_forward():
            robot.forward()
    robot.turn_right()

print()
print(f'Final position: ({robot.col}, {robot.row}) — goal is (0, 2)')
print(f'Total moves taken: {robot.steps}')
print(f'Goal reached? {(robot.col, robot.row) == (0, 2)}')

### Recap

| Pattern | Game block | Python | When to use |
| --- | --- | --- | --- |
| Sequence | stacked blocks | one statement per line | always — it's the default |
| Iteration | `Repeat N times` | `for _ in range(N):` | doing the same thing N times |
| Selection | `If Can Move Forward` | `if condition:` | only doing something *if* it's safe / valid |

Every level in the game can be solved with combinations of these three. Same is true of every Python script you'll ever write.

## Hacks (Challenges)

1. **3-Star Sweep.** Earn 3 stars on every level. Stars are saved between sessions — open the level list to see your record.
2. **Manual Mode.** Solve Level 6 *without* using `If Can Move Forward`. How many more blocks did you need? Why?
3. **Predict the Output.** Before running the cell below, write down where you think the robot will end up. Then run it.
4. **Design Your Own Level.** Open `_projects/robot-logic-game/levels/levelConfigs.js` and add a new entry. Try a 9×9 grid, a few wall clusters, and tune `starThresholds` so the optimal solution earns 3 stars.

In [ ]:
# Predict-the-output challenge — what's the robot's final position?
# (Cover the cell output before running.)
robot = Robot(col=3, row=3, facing=0)

for i in range(4):
    for _ in range(i + 1):
        robot.forward()
    robot.turn_right()

print()
print(f'Final: ({robot.col}, {robot.row}) facing {robot.NAMES[robot.facing]}')

## Reflection

Take a moment with these questions — they're more important than the code.

1. **Compression.** Which pattern (sequence, loops, conditionals) gave you the biggest reduction in block count? Why?
2. **Translation.** Pick one of your game programs and rewrite it as Python using the `Robot` class above. What did you have to change? What stayed the same?
3. **Generalization.** The block `Repeat N times` always runs *exactly* N times. What would you call a loop that runs *while* something is true and stops when it isn't? (Hint: it's also a Python keyword — and a natural next addition to the game.)
4. **Real-world parallel.** Think of one routine you do every day. Identify the sequence, the loops, and the conditionals.